# Non-prior, Prior, Last-year 돌린 후 FAILED까지 수기처리 후 돌릴 것.

In [1]:
# ─────────────────────────────────────────────────────────────────
# [셀 2] POST-PROCESSING v3.2: 매핑 + long 변환 + currency 환율 적용
#        + channel: US 매핑 / dummy 0 / PAID_NONPAID
#        + metric_col 키 분리 (J1~J7, J11)
#        + 개별 _stacked_separate.csv + 전체 union 저장
#
# ──────── 변경 이력 (v3 → v3.2) ────────────────────────────────────
# 수정일: 2026-04-10 / 작성자: Jonghyun Park w/claude
#
# [FIX-9] _TS_PAT 타임스탬프 정규식 수정 (\d{6} → \d{4})
#   - 실제 파일 타임스탬프가 HHMM(4자리)인데 HHMMSS(6자리)로 매칭하여
#     deduplication 실패 → 동일 tb_key 파일이 2개일 때 모두 처리됨
#   - 수정: \d{6} → \d{4} → 최신 파일 1개만 선택, union 중복 삽입 방지
#
# [FIX-10] _is_channel_table — 3_1_channel_external 포함
#   - FIX-2에서 3_2/7_1만 channel로 제한하면서 3_1_channel_external이
#     non-channel로 처리 → value 컬럼(채널명) 무시 → ITEM=NaN
#   - 수정: 패턴에 3_1_channel_external 추가 → 채널명 ITEM 정상 출력
#
# [FIX-11] _is_channel_table — 리포트 번호 하드코딩 제거, 매핑 CSV 기반 판별
#   - 기존: 파일명의 3_2_channel/7_1/3_1_channel_external 패턴으로 판별
#   - 문제: 3_2_order_cvr_smartthings처럼 3_2가 포함되면 channel로 오판
#           → value 컬럼(Mobile Application)이 ITEM으로 들어감 (의도: smartthings)
#   - 변경: 매핑 column명 중 하나라도 "_"로 끝나면 channel 테이블로 판별
#           → 리포트 번호 무관, 매핑 CSV 기준 자동 판별
#   - 함수 시그니처 변경: _is_channel_table(tb_key, mapping_df)
#
# ──────── 변경 이력 (v2 → v3) ─────────────────────────────────────
# 수정일: 2026-03-31 / 작성자: Jonghyun Park w/claude
#
# [FIX-5] 더미 삽입 US 포함 여부 확인
#   - 확인 결과: _global_sites -= _us_sites 로직으로 이미 올바르게 분리됨
#   - 글로벌 더미에 US site 미포함, US 더미에만 US site 적용 확인
#   - Pre-scan 후 Global/US 중복 여부 검증 로그 추가
#
# [FIX-6] App 없는 site의 App 데이터 0처리 (신규)
#   - ref/app_O_X.csv B열=X인 site를 _no_app_sites로 로드
#   - _apply_j_cols 이후 J5(DEVICE TYPE)가 app/android/ios인 행의
#     metric_value_origin / metric_value_adj 를 0.0으로 처리 (행 유지)
#   - 대소문자 무관 비교 (str.lower())
#
# [FIX-7] Revenue 환율 적용 기준 연도를 End_Date 기준으로 변경
#   - 기존: pd.to_datetime(Start_Date).dt.year
#   - 변경: pd.to_datetime(End_Date).dt.year
#   - 이유: 기간이 연도를 걸치는 경우 end date 연도의 환율을 적용
#
# [FIX-8] Union 레벨 US dummy 삽입 로직 수정
#   - 기존: _us_table_cols(US 매핑에 있는 column)이면 dummy skip
#   - 문제: US 테이블이 매핑에 있어도 실제 파일이 없거나 FAILED이면
#           _us_mc에 없는데도 "이미 처리됨"으로 판단해 dummy 미삽입
#           예) 4_1 US 버전이 매핑에 있으나 데이터 없을 때 main_then_pd_all 누락
#   - 수정: _us_table_cols 조건 제거 → _missing_for_us(= _non_us_mc - _us_mc)만으로 판단
#           US 전용 column은 _non_us_mc에 없으므로 _missing_for_us에 포함되지 않아 문제없음
#
# ──────── 변경 이력 (v1 → v2) ─────────────────────────────────────
# [FIX-1] is_us_table 판별 로직 수정
#   - 기존: tb_key.startswith("us_")
#   - 변경: _is_us_table(tb_key) 함수 사용
#           → "us_" 또는 "last_us_"로 시작하는 테이블 모두 US로 인식
#   - 영향: Pre-scan에서 US site가 _global_sites에 포함되던 버그 해소
#           → 글로벌 테이블에서 US dummy 0이 생성되지 않음
#           → last_us_ 테이블에서 non-US site dummy 0이 생성되지 않음
#   - 적용 위치: Pre-scan, 메인 처리 루프, channel 후처리
#
# [FIX-2] is_channel 판별 로직 수정
#   - 기존: "channel" in tb_key.lower()
#   - 변경: _is_channel_table(tb_key) 함수 사용
#           → column이 trailing "_"로 끝나는 3_2, 7_1 계열만 channel
#           → 3_1_channel_internal 등은 non-channel로 처리
#   - 이유: 3_1은 column명에 이미 채널이 내장되어 있어 dimension breakdown 불필요
#           channel 로직의 mc_has_ch 경로에서 existing 체크 시
#           value=None vs 실제 value 불일치로 불필요한 dummy 0 생성됨
#   - 적용 위치: 메인 처리 루프 is_channel 판별부
#
# [FIX-3] Global ↔ US 테이블 column 중복 제거
#   - 기존: 글로벌 테이블과 us_ 테이블이 같은 column key를 생산하여
#           union 시 동일 KEY가 2행(둘 다 실데이터 또는 실데이터+dummy)
#   - 변경: 글로벌 테이블 처리 시 US site 데이터를 제외
#           → us_ 테이블에서만 US 데이터를 가져옴
#   - 적용 위치: 메인 처리 루프에서 melt 후 US 행 필터링
#
# [FIX-4] Union 레벨 US dummy 삽입 로직 개선
#   - 기존: non-US에 있고 US에 없는 metric_col에 무조건 0행 삽입
#           → US 전용 테이블(us_, last_us_)의 column과 글로벌 전용 column이
#             서로 다른 경우에도 dummy 생성하여 불필요한 행 증가
#   - 변경: US 전용 metric_col 패턴과 글로벌 전용 패턴을 구분하여
#           실제로 US 리포트에 포함되어야 하는 column만 dummy 삽입
#           → US 테이블에서 이미 처리된 column은 skip
# ─────────────────────────────────────────────────────────────────

import re
from pathlib import Path
from datetime import datetime
import pandas as pd

MAPPING_CSV  = Path.cwd().parent / "ref" / "tb_column_name_mapping.csv"
CURRENCY_CSV = Path.cwd().parent / "ref" / "currency.csv"
EXPORTS_DIR  = Path.cwd().parent / "aa_exports"

# 리포트 번호 패턴 (1_1, 3_2 등): 이 패턴이 있는 파일만 dummy 0 삽입 / CSV 저장 / union 포함
_REPORT_NO_PAT = re.compile(r'(?:^|_)\d{1,2}_\d{1,2}(?:_|$)')

# ─────────────────────────────────────────────────────────────────
# [FIX-1] US 테이블 판별 함수
# ─────────────────────────────────────────────────────────────────
def _is_us_table(tb_key: str) -> bool:
    """us_ 또는 last_us_ 로 시작하는 테이블을 US 테이블로 판별"""
    return tb_key.startswith("us_") or tb_key.startswith("last_us_")

# ─────────────────────────────────────────────────────────────────
# [FIX-2/11] Channel 테이블 판별 함수
# ─────────────────────────────────────────────────────────────────
# 매핑 column명이 "_"로 끝나는 경우 value 컬럼에 채널명이 있음 → channel 테이블
# 리포트 번호(3_2, 7_1 등)에 의존하지 않고 매핑 CSV 기준으로 자동 판별
# ex) 3_2_channel_order_cvr: ...null_  → channel ✓
#     3_2_order_cvr_smartthings: ...null_smartthings → non-channel ✓
#     3_1_channel_external: ...null_  → channel ✓
#     3_1_external_smartthings: ...null_smartthings → non-channel ✓

def _is_channel_table(tb_key: str, mapping_df: "pd.DataFrame") -> bool:
    """매핑 column명 중 하나라도 "_"로 끝나면 channel 테이블 (value 컬럼 = 채널명)"""
    tb_map = mapping_df[mapping_df["tb"] == tb_key]  # 정확한 tb_key 매핑 조회
    if tb_map.empty:  # _prior 등 suffix로 매핑 없는 경우 suffix 제거 후 재조회
        tb_map = mapping_df[mapping_df["tb"] == re.sub(r"_prior$", "", tb_key)]
    # column명이 "_"로 끝남 = metric_col이 "_"로 끝남 = value 컬럼에 채널명 있음
    # ex) "3_2_all_2026_cmp_pc_entry_null_"            → True  (채널별 breakdown)
    # ex) "3_2_all_2026_cmp_pc_entry_null_smartthings" → False (아이템 내장)
    return bool(tb_map["column"].str.endswith("_").any())
cur_df = pd.read_csv(CURRENCY_CSV)
col_latest = cur_df.columns[2]
col_prior  = cur_df.columns[3]
year_latest = str(pd.to_datetime(col_latest).year)
year_prior  = str(pd.to_datetime(col_prior).year)
print(f"▶ 환율 컬럼: {col_latest}({year_latest}) / {col_prior}({year_prior})")

cur_map = {
    str(r["site_code"]).strip().lower(): {
        year_latest: r[col_latest],
        year_prior:  r[col_prior],
    }
    for _, r in cur_df.iterrows()
}

# ── app_O_X 로드 (App 없는 site 목록) ──────────────────────────────
# [FIX-6] ref/app_O_X.csv: A열=site_code, B열=O(앱있음)/X(앱없음)
APP_OX_CSV = Path.cwd().parent / "ref" / "app_O_X.csv"
_app_ox_df  = pd.read_csv(APP_OX_CSV, encoding="utf-8-sig")
_app_ox_site_col = _app_ox_df.columns[0]   # A열 = site_code
_app_ox_flag_col = _app_ox_df.columns[1]   # B열 = O/X
_no_app_sites = set(
    _app_ox_df.loc[
        _app_ox_df[_app_ox_flag_col].astype(str).str.strip().str.upper() == "X",
        _app_ox_site_col
    ].str.strip().str.lower()
)
print(f"\u25b6 App 없는 site ({len(_no_app_sites)}개): {sorted(_no_app_sites)}")

# ── channel 매핑 정의 ─────────────────────────────────────────────
us_mapping = {
    "US Channel":"Global Channel Mapping ",
    "Affiliate":"Affiliate Marketing",
    "Display":"Display",
    "Display Retargeting":"Display",
    "Other External Campaign":"Paid Others",
    "Other Paid Ecomm":"Paid Others",
    "Paid Search":"Paid Search",
    "Paid Search - eComm":"Paid Search",
    "PLA":"Pmax",
    "Social (Paid)":"Social Media Campaigns",
    "Vanity":"Paid Others",
    "Direct":"Direct",
    "Email - CRM":"Email",
    "Email - eComm":"Email",
    "Email - Upsell it":"Email",
    "Email (Retired)":"Email",
    "EPP":"EPP - US",
    "Natural Search":"Natural Search",
    "Other":"Other",
    "None":"Other",
    "Push Notifications":"Push",
    "Referring Domains":"Other",
    "Session Refresh":"Session Refresh",
    "Social (Free and Owned)":"Owned Social",
    "Social (Retired)":"Social Network Referrals",
    "Other External CampaignSegments":"Mobile Application",
    "Other External CampaignUS_Smartthings":"Mobile Application - Smartthings",
    "Other External CampaignUS_Samsung Members":"Mobile Application - Samsung Members",
    "SMS":"SMS",
    "Gen AI Search":"Gen AI (Organic)"
}

global_paid_mapping = {
    "Paid Search":"Paid",
    "Social Media Campaigns":"Paid",
    "Affiliate Marketing":"Paid",
    "Display":"Paid",
    "Other Campaigns":"Paid",
    "QR code (Paid)":"Paid",
    "Pmax":"Paid",
    "Demand Gen":"Paid",
    "Paid Others":"Paid",
    "Video":"Paid",
    "Gen AI (Paid)":"Paid",
    "Session Refresh":"Non-Paid",
    "Email":"Non-Paid",
    "Direct":"Non-Paid",
    "Natural Search":"Non-Paid",
    "Mobile Application":"Non-Paid",
    "Push":"Non-Paid",
    "Social Network Referrals":"Non-Paid",
    "Other":"Non-Paid",
    "SMS":"Non-Paid",
    "Owned Social":"Non-Paid",
    "QR code (Owned)":"Non-Paid",
    "Samsung Web":"Non-Paid",
    "Gen AI (Organic)":"Non-Paid",
    "Owned Others":"Non-Paid"
}

# ── 최종 컬럼 정리 ────────────────────────────────────────────────
def finalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "PAID_NONPAID" not in df.columns:
        df["PAID_NONPAID"] = "-"
    df["TIER"]  = ""
    df["공란1"] = ""
    df["공란2"] = ""
    df["공란3"] = ""
    df["공란4"] = ""
    if "Site_Code" in df.columns and "metric_col" in df.columns:
        df["metric_col"] = df["Site_Code"].str.strip().str.lower() + "_" + df["metric_col"].astype(str)
    df = df.rename(columns={
        "Subsidiary":         "SUBS",
        "Country":            "COUNTRY",
        "Site_Code":          "SITE CODE",
        "J11":                "REPORT NO.",
        "J3":                 "DIVISION",
        "J4":                 "DATE",
        "J5":                 "DEVICE TYPE",
        "J6":                 "TYPE",
        "J7":                 "LOGIN/NON",
        "PAID_NONPAID":       "PAID/NONPAID",
        "value":              "ITEM",
        "metric_value_adj":   "VALUE",
        "metric_col":         "KEY",
        "metric_value_origin":"value_origin",
        "Start_Date":         "start_date",
        "End_Date":           "end_date",
    })
    final_cols = [
        "TIER", "SUBS", "COUNTRY", "SITE CODE",
        "REPORT NO.", "DIVISION", "DATE", "DEVICE TYPE", "TYPE", "LOGIN/NON",
        "PAID/NONPAID", "ITEM", "VALUE", "KEY",
        "공란1", "공란2", "공란3", "공란4",
        "value_origin", "start_date", "end_date"
    ]
    return df[[c for c in final_cols if c in df.columns]]


# ── 캠페인 키 분리 유틸 ───────────────────────────────────────────
report_no_mapping = {
    "1_1": "1. Campaign Basic Traffic",  # basic_traffic_cmp
    "2_1": "2. Campaign Time Spent Per Visit By Country",  # basic_traffic_time_cmp
    "3_1": "3-1. Campaign Internal/External Traffic By Channel",  # channel_external_cmp / channel_internal_cmp / external_smartthings_cmp
    "3_2": "3-2. Campaign Order Conversion/Traffic By Channel",  # channel_order_cvr_cmp / order_cvr_smartthings_cmp
    "3_3": "3-3. Campaign Home Kv & Offer Kv Clicks",  # home_kv_offer_kv_to_cmp
    "4_1": "4-1. Campaign Order Conversion Funnel",  # order_cvr_funnel_cmp
    "4_2": "4-2. Campaign Order Conversion Funnel (Login/Non-Login)",  # order_cvr_funnel_login_cmp / order_cvr_funnel_logout_cmp
    "5_1": "5. Campaign Order Conversion",  # pf_order_cvr_cmp
}

def _split_by_number_pattern(parts):
    if len(parts) >= 2 and re.fullmatch(r"\d{1,2}", parts[0]) and re.fullmatch(r"\d{1,2}", parts[1]):
        return "", f"{parts[0]}_{parts[1]}", parts[2:]
    for i in (2, 3):
        if i + 1 < len(parts) and parts[i].isdigit() and parts[i+1].isdigit():
            return '_'.join(parts[:i]), f"{parts[i]}_{parts[i+1]}", parts[i+2:]
    return parts[0], f"{parts[1]}_{parts[2]}", parts[3:]

def _find_report_key(values):
    for v in values:
        if isinstance(v, str) and re.fullmatch(r"\d+_\d+", v):
            return v
    return ''

def split_metric_col(val: str) -> dict:
    parts = str(val).split('_')
    if len(parts) < 3:
        j1_to_j7 = (parts + [''] * 7)[:7]
        return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": "", "metric_name": ""}
    first, second, remaining = _split_by_number_pattern(parts)
    processed = []
    i = 0
    while i < len(remaining):
        if remaining[i].isdigit() and remaining[i].startswith('202'):
            if i + 2 < len(remaining) and remaining[i+2].lower() == 'prior':
                processed.append(f"{remaining[i]}_{remaining[i+1]}_{remaining[i+2]}")
                i += 3
            elif i + 1 < len(remaining):
                processed.append(f"{remaining[i]}_{remaining[i+1]}")
                i += 2
            else:
                processed.append(remaining[i])
                i += 1
        else:
            processed.append(remaining[i])
            i += 1
    all_parts = [first, second] + processed
    j1_to_j7 = (all_parts + [''] * 7)[:7]
    metric_name = '_'.join(all_parts[7:]) if len(all_parts) > 7 else ''
    report_key = _find_report_key(j1_to_j7)
    return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": report_no_mapping.get(report_key, ''), "metric_name": metric_name}

def _apply_j_cols(df: pd.DataFrame) -> pd.DataFrame:
    records = [split_metric_col(v) for v in df["metric_col"]]
    if not records:
        extra_cols = [f"J{i+1}" for i in range(7)] + ["J11", "metric_name"]
        df = df.copy()
        for col in extra_cols:
            df[col] = pd.NA
        return df
    j_df = pd.DataFrame(records, index=df.index)
    return pd.concat([df, j_df], axis=1)

# ── non-channel dummy 0 삽입 유틸 ────────────────────────────────
def _insert_nonchannel_dummy(df_long: pd.DataFrame, tb_key: str,
                              master_sites: set, site_meta_map: dict) -> pd.DataFrame:
    """마스터 site_code 기준으로 누락된 site×metric_col 조합에 0행 삽입"""
    all_sites   = list(master_sites)
    metric_cols = df_long["metric_col"].unique()

    meta_cols = [c for c in ["Subsidiary", "Country", "RSID"] if c in df_long.columns]
    file_meta = (
        df_long.groupby("Site_Code")[meta_cols]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .reset_index()
    )
    date_cols = [c for c in ["Start_Date", "End_Date", "itemId"] if c in df_long.columns]
    file_date_meta = (
        df_long.groupby("Site_Code")[date_cols]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .reset_index()
    )
    file_date_mode = {
        c: df_long[c].mode().iloc[0] if not df_long[c].mode().empty else None
        for c in date_cols
    }

    cross = pd.MultiIndex.from_product(
        [all_sites, metric_cols],
        names=["Site_Code", "metric_col"]
    ).to_frame(index=False)

    existing = df_long[["Site_Code", "metric_col"]].drop_duplicates()
    existing["_exists"] = True

    merged = cross.merge(existing, on=["Site_Code", "metric_col"], how="left")
    dummy_rows = merged[merged["_exists"].isna()].drop(columns=["_exists"])

    if dummy_rows.empty:
        return df_long

    dummy_rows = dummy_rows.merge(file_meta, on="Site_Code", how="left")
    dummy_rows = dummy_rows.merge(file_date_meta, on="Site_Code", how="left")

    for _col in meta_cols:
        if _col not in dummy_rows.columns:
            continue
        _na_mask = dummy_rows[_col].isna()
        if _na_mask.any():
            dummy_rows.loc[_na_mask, _col] = dummy_rows.loc[_na_mask, "Site_Code"].map(
                lambda s: site_meta_map.get(s, {}).get(_col)
            )

    for _col, _val in file_date_mode.items():
        if _col in dummy_rows.columns:
            dummy_rows[_col] = dummy_rows[_col].fillna(_val)

    dummy_rows["metric_value_origin"] = 0.0
    dummy_rows["metric_value_adj"]    = 0.0
    dummy_rows = _apply_j_cols(dummy_rows)
    dummy_rows["value"] = dummy_rows["metric_name"]
    df_long = pd.concat([df_long, dummy_rows], ignore_index=True)
    print(f"  {tb_key} → dummy {len(dummy_rows)}행 삽입")
    return df_long

# ── 매핑 로드 ─────────────────────────────────────────────────────
mapping_df = pd.read_csv(MAPPING_CSV)
id_vars = ["Subsidiary", "Country", "Site_Code", "RSID",
           "Start_Date", "End_Date", "itemId", "value"]

ok_list, skip_list, no_vars_list, no_report_list = [], [], [], []
channel_frames = {}
all_frames = []

# ── [260316 개선] tb_key별 최신 파일만 선택 ──────────────────────
_all_csv = [f for f in EXPORTS_DIR.glob("*.csv")
            if "_long" not in f.name
            and "_stacked" not in f.name
            and not f.name.startswith("union_")]
_TS_PAT = re.compile(r"_(\d{8}_\d{4})$")

def _get_ts(p):
    m = _TS_PAT.search(p.stem)
    return m.group(1) if m else ""

_latest_map = {}
for _f in _all_csv:
    _key = _TS_PAT.sub("", _f.stem)
    if _key not in _latest_map or _get_ts(_f) > _get_ts(_latest_map[_key]):
        _latest_map[_key] = _f

csv_files = sorted(_latest_map.values())
print(f"▶ 처리 대상: {len(csv_files)}개 파일 (tb_key별 최신 파일 기준)\n")

# ── Pre-scan: 전체 site_code 및 메타 정보 수집 ───────────────────
# [FIX-1] _is_us_table() 사용
_META_COLS = ["Subsidiary", "Country", "RSID"]
_global_sites = set()
_us_sites     = set()
_site_meta_map = {}

for _path in csv_files:
    _tk = re.sub(r"_\d{8}_\d{6}$", "", _path.stem)
    try:
        _usecols = lambda c: c in (["Site_Code"] + _META_COLS)
        _tmp = pd.read_csv(_path, encoding="utf-8-sig", usecols=_usecols)
        for _sc_raw in _tmp["Site_Code"].dropna().unique():
            _sc = str(_sc_raw).strip()
            if _sc not in _site_meta_map:
                _row = _tmp[_tmp["Site_Code"].astype(str).str.strip() == _sc].iloc[0]
                _site_meta_map[_sc] = {c: _row[c] for c in _META_COLS if c in _tmp.columns}
            # ── [FIX-1] last_us_ 도 US 테이블로 분류 ──
            if _is_us_table(_tk):
                _us_sites.add(_sc)
            else:
                _global_sites.add(_sc)
    except Exception:
        pass

# ── [FIX-1 보완] _global_sites에서 US 전용 site 제거 ──
# last_us_ 버그가 수정되었으므로 US site가 _global_sites에 포함되지 않지만,
# 혹시 글로벌 테이블 CSV에 US 행이 포함된 경우를 대비한 안전장치
_global_sites -= _us_sites

print(f"▶ Pre-scan 완료")
print(f"  Global site_codes ({len(_global_sites)}개): {sorted(_global_sites)}")
print(f"  US site_codes ({len(_us_sites)}개): {sorted(_us_sites)}")
# [FIX-5] Global/US site 중복 없음 검증 (더미 삽입 기준 올바름 확인)
_overlap = _global_sites & _us_sites
if _overlap:
    print(f"  [WARN] Global/US site 중복 감지: {_overlap} \u2192 _global_sites에서 이미 제거됨")
else:
    print(f"  [FIX-5 확인] Global/US site 중복 없음 \u2192 더미 삽입 기준 올바름 \u2713")
print()

# ── 메인 처리 루프 ────────────────────────────────────────────────
for src_path in sorted(csv_files):
    stem   = src_path.stem
    tb_key = re.sub(r"_\d{8}_\d{4}$", "", stem)

    if not _REPORT_NO_PAT.search(tb_key) and not _is_channel_table(tb_key, mapping_df):
        no_report_list.append(tb_key)
        continue

    tb_mapping = mapping_df[mapping_df["tb"] == tb_key]
    if tb_mapping.empty:
        tb_mapping = mapping_df[mapping_df["tb"] == re.sub(r"_prior$", "", tb_key)]

    if tb_mapping.empty:
        skip_list.append(tb_key)
        continue

    rename_dict = dict(zip(tb_mapping["value_n"], tb_mapping["column"]))

    df = pd.read_csv(src_path, encoding="utf-8-sig")
    df = df.rename(columns=rename_dict)
    value_vars = [c for c in rename_dict.values() if c in df.columns]

    if not value_vars:
        no_vars_list.append(tb_key)
        continue

    for v in value_vars:
        df[v] = pd.to_numeric(
            df[v].astype(str).str.strip().str.replace("'", "", regex=False),
            errors="coerce"
        )

    df_long = df.melt(
        id_vars=[c for c in id_vars if c in df.columns],
        value_vars=value_vars,
        var_name="metric_col",
        value_name="metric_value_origin"
    )

    # ── [FIX-3] 글로벌 테이블에서 US site 데이터 제외 ─────────────
    # US 데이터는 us_/last_us_ 전용 테이블에서만 가져옴
    # 글로벌 테이블에 US 행이 포함되어 있으면 제거하여 KEY 중복 방지
    if not _is_us_table(tb_key) and _us_sites:
        _us_in_global = df_long["Site_Code"].str.strip().isin(_us_sites)
        if _us_in_global.any():
            _removed = _us_in_global.sum()
            df_long = df_long[~_us_in_global].reset_index(drop=True)
            print(f"  {tb_key} → 글로벌 테이블에서 US site {_removed}행 제외 (us_ 테이블에서 처리)")

    # ── currency 환율 적용 ────────────────────────────────────────
    df_long["_year"] = pd.to_datetime(df_long["End_Date"]).dt.year.astype(str)  # [FIX-7] End_Date 기준
    df_long["_site"] = df_long["Site_Code"].str.strip().str.lower()

    def calc_adj(row):
        if "revenue" not in str(row["metric_col"]).lower():
            return row["metric_value_origin"]
        rates = cur_map.get(row["_site"])
        if not rates:
            return row["metric_value_origin"]
        rate = rates.get(row["_year"])
        if rate is None:
            return row["metric_value_origin"]
        try:
            return float(row["metric_value_origin"]) * float(rate)
        except (TypeError, ValueError):
            return row["metric_value_origin"]

    df_long["metric_value_adj"] = df_long.apply(calc_adj, axis=1)
    df_long = df_long.drop(columns=["_year", "_site"])

    df_long["metric_value_origin"] = pd.to_numeric(
        df_long["metric_value_origin"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )
    df_long["metric_value_adj"] = pd.to_numeric(
        df_long["metric_value_adj"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )

    # ── metric_col 키 분리 ────────────────────────────────────────
    df_long = _apply_j_cols(df_long)

    # ── [FIX-6] App 없는 site: J5(DEVICE TYPE)가 app/android/ios인 행 0처리 ──
    # app_O_X.csv B열 X인 site에서 앱 관련 device type 데이터 값을 0으로 처리
    _APP_DEVICE_TYPES = {"app", "android", "ios"}
    if "J5" in df_long.columns and _no_app_sites:
        _app_zero_mask = (
            df_long["Site_Code"].str.strip().str.lower().isin(_no_app_sites)
            & df_long["J5"].astype(str).str.lower().isin(_APP_DEVICE_TYPES)
        )
        if _app_zero_mask.any():
            _cnt = _app_zero_mask.sum()
            df_long.loc[_app_zero_mask, "metric_value_origin"] = 0.0
            df_long.loc[_app_zero_mask, "metric_value_adj"]    = 0.0
            print(f"  {tb_key} \u2192 App 없는 site App 데이터 {_cnt}행 0처리")

    # ── [FIX-2] channel 판별: _is_channel_table() 사용 ────────────
    is_channel = _is_channel_table(tb_key, mapping_df)
    if is_channel:
        channel_frames[tb_key] = df_long
    else:
        # ── year-split 합산 ───────────────────────────────────────
        _yr_mask = df_long["value"].astype(str).str.match(r'^20\d{2}', na=False)
        if _yr_mask.any():
            _agg_keys = list(dict.fromkeys(
                [c for c in
                 ["Subsidiary", "Country", "Site_Code", "RSID", "Start_Date", "End_Date",
                  "metric_col"] + [f"J{i+1}" for i in range(7)] + ["J11", "metric_name"]
                 if c in df_long.columns]
            ))
            _yr_df = df_long[_yr_mask]
            _time_mask = _yr_df["metric_col"].str.contains("time", case=False, na=False)

            def _do_agg(sub_df, use_mean):
                _agg = {"metric_value_origin": "mean" if use_mean else "sum",
                        "metric_value_adj":    "mean" if use_mean else "sum"}
                if "itemId" in sub_df.columns:
                    _agg["itemId"] = "first"
                return sub_df.groupby(_agg_keys, as_index=False).agg(_agg)

            _parts = []
            if _time_mask.any():
                _parts.append(_do_agg(_yr_df[_time_mask], use_mean=True))
            if (~_time_mask).any():
                _parts.append(_do_agg(_yr_df[~_time_mask], use_mean=False))

            _summed = pd.concat(_parts, ignore_index=True) if _parts else _yr_df.iloc[0:0]
            df_long = pd.concat([df_long[~_yr_mask], _summed], ignore_index=True)
            print(f"  {tb_key} → year-split: {_yr_mask.sum()}행 → {len(_summed)}행 (time계열 평균, 나머지 합산)")

        df_long["value"] = df_long["metric_name"]

        # ── [FIX-1] master site 결정: _is_us_table() 사용 ─────────
        master = _us_sites if _is_us_table(tb_key) else _global_sites
        df_long = _insert_nonchannel_dummy(df_long, tb_key, master, _site_meta_map)
        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
        df_long.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        all_frames.append(df_long)

# ── channel 후처리 (루프 종료 후) ────────────────────────────────
if channel_frames:
    print(f"\n▶ channel 후처리: {list(channel_frames.keys())}")

    for tb_key, df_long in channel_frames.items():

        if "value" in df_long.columns:
            df_long["value"] = df_long["value"].astype(object)

        # 1. US value 치환
        is_us = df_long["Site_Code"].str.strip().str.lower() == "us"
        df_long.loc[is_us, "value"] = (
            df_long.loc[is_us, "value"].map(us_mapping).fillna(df_long.loc[is_us, "value"])
        )

        # 2. dummy 0 삽입 (마스터 site_code × 글로벌 채널 기준)
        # ── [FIX-1] master site 결정: _is_us_table() 사용 ─────────
        master = _us_sites if _is_us_table(tb_key) else _global_sites
        all_sites = list(master)

        _channel_master = set(global_paid_mapping.keys())
        all_values = list(set(df_long["value"].dropna().unique()) | _channel_master)

        metric_cols = df_long["metric_col"].unique()
        mc_needs_ch = [mc for mc in metric_cols if str(mc).endswith("_")]
        mc_has_ch   = [mc for mc in metric_cols if not str(mc).endswith("_")]

        meta_cols = [c for c in ["Subsidiary", "Country", "RSID"] if c in df_long.columns]
        meta_df = (
            df_long.groupby("Site_Code")[meta_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )
        date_cols = [c for c in ["Start_Date", "End_Date", "itemId"] if c in df_long.columns]
        date_meta = (
            df_long.groupby("Site_Code")[date_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )
        file_date_mode = {
            c: df_long[c].mode().iloc[0] if not df_long[c].mode().empty else None
            for c in date_cols
        }

        cross_parts = []
        if mc_needs_ch and all_values:
            cross_parts.append(
                pd.MultiIndex.from_product(
                    [all_sites, all_values, mc_needs_ch],
                    names=["Site_Code", "value", "metric_col"]
                ).to_frame(index=False)
            )
        if mc_has_ch:
            _no_ch = pd.MultiIndex.from_product(
                [all_sites, mc_has_ch],
                names=["Site_Code", "metric_col"]
            ).to_frame(index=False)
            _no_ch["value"] = None
            cross_parts.append(_no_ch)

        if cross_parts:
            cross = pd.concat(cross_parts, ignore_index=True)

            existing = df_long[["Site_Code", "value", "metric_col"]].drop_duplicates()
            existing["_exists"] = True

            merged = cross.merge(existing, on=["Site_Code", "value", "metric_col"], how="left")
            dummy_rows = merged[merged["_exists"].isna()].drop(columns=["_exists"])

            if not dummy_rows.empty:
                dummy_rows = dummy_rows.merge(meta_df, on="Site_Code", how="left")
                dummy_rows = dummy_rows.merge(date_meta, on="Site_Code", how="left")
                for _col in meta_cols:
                    if _col not in dummy_rows.columns:
                        continue
                    _na_mask = dummy_rows[_col].isna()
                    if _na_mask.any():
                        dummy_rows.loc[_na_mask, _col] = dummy_rows.loc[_na_mask, "Site_Code"].map(
                            lambda s: _site_meta_map.get(s, {}).get(_col)
                        )
                for _col, _val in file_date_mode.items():
                    if _col in dummy_rows.columns:
                        dummy_rows[_col] = dummy_rows[_col].fillna(_val)
                dummy_rows["metric_value_origin"] = 0.0
                dummy_rows["metric_value_adj"]    = 0.0
                dummy_rows = _apply_j_cols(dummy_rows)
                df_long = pd.concat([df_long, dummy_rows], ignore_index=True)
                print(f"  {tb_key} → dummy {len(dummy_rows)}행 삽입")

        # 2-후처리. "None" 채널명 → "Other"
        _none_mask = df_long["value"].astype(str) == "None"
        if _none_mask.any():
            df_long.loc[_none_mask, "value"] = "Other"

        # 2-후처리2. mc_needs_ch 행 중 value=NaN인 행 제외
        _mc_needs_ch_mask = df_long["metric_col"].astype(str).str.endswith("_")
        _val_missing_mask = df_long["value"].isna()
        _drop_mask = _mc_needs_ch_mask & _val_missing_mask
        if _drop_mask.any():
            print(f"  {tb_key} → 채널명 NaN 행 {_drop_mask.sum()}개 제외: "
                  f"{df_long.loc[_drop_mask, 'Site_Code'].value_counts().to_dict()}")
            df_long = df_long[~_drop_mask].reset_index(drop=True)

        # 3. metric_col에 채널명 합치기
        _has_year    = df_long["value"].astype(str).str.contains(r'\b20\d{2}\b', regex=True, na=False)
        _mc_needs_ch = df_long["metric_col"].astype(str).str.endswith("_")
        _mc_has_ch_rows = ~_mc_needs_ch & ~_has_year

        df_long["metric_col"] = df_long["metric_col"].astype(str).where(
            _has_year | ~_mc_needs_ch,
            df_long["metric_col"].astype(str) + df_long["value"].astype(str)
        )

        # 4. value(ITEM) 보정
        df_long.loc[_has_year, "value"] = df_long.loc[_has_year, "metric_name"].str.rstrip("_")
        _value_null = df_long["value"].isna() | (df_long["value"].astype(str) == "None")
        _mc_has_ch_null = _mc_has_ch_rows & _value_null
        if _mc_has_ch_null.any():
            _emb = df_long.loc[_mc_has_ch_null, "metric_col"].astype(str).str.extract(
                r'_null_(.+)$', expand=False
            )
            _valid = _emb.notna()
            if _valid.any():
                df_long.loc[_emb[_valid].index, "value"] = _emb[_valid].values
        _value_null2 = df_long["value"].isna() | (df_long["value"].astype(str) == "None")
        df_long.loc[_value_null2, "value"] = df_long.loc[_value_null2, "metric_name"].str.rstrip("_")

        # 5. PAID_NONPAID 부여
        _ch_for_paid = df_long["value"].copy()
        if _mc_has_ch_rows.any():
            _emb_paid = df_long.loc[_mc_has_ch_rows, "metric_col"].astype(str).str.extract(
                r'_null_(.+)$', expand=False
            )
            _valid_paid = _emb_paid.notna()
            if _valid_paid.any():
                _ch_for_paid.loc[_emb_paid[_valid_paid].index] = _emb_paid[_valid_paid].values

        df_long["PAID_NONPAID"] = _ch_for_paid.map(global_paid_mapping)
        val_idx = df_long.columns.get_loc("value")
        cols = list(df_long.columns)
        cols.remove("PAID_NONPAID")
        cols.insert(val_idx, "PAID_NONPAID")
        df_long = df_long[cols]

        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
        finalize_df(df_long).to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        if _REPORT_NO_PAT.search(tb_key):
            all_frames.append(df_long)

# ── union 생성 ────────────────────────────────────────────────────
if all_frames:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    union_path = EXPORTS_DIR / f"union_{ts}.csv"
    union_df = pd.concat(all_frames, ignore_index=True)
    if "PAID_NONPAID" in union_df.columns:
        union_df["PAID_NONPAID"] = union_df["PAID_NONPAID"].fillna("-")

    # ── [FIX-4] US dummy 보완: 개선된 로직 ────────────────────────
    # US 전용 테이블에서 이미 처리된 metric_col은 건드리지 않음
    # 글로벌 전용 column 중 US에 없는 것만 dummy 삽입
    _non_us_mask = ~union_df["Site_Code"].str.strip().isin(_us_sites)
    _us_mask     = union_df["Site_Code"].str.strip().isin(_us_sites)
    _non_us_mc   = set(union_df.loc[_non_us_mask, "metric_col"].dropna().unique())
    _us_mc       = set(union_df.loc[_us_mask,     "metric_col"].dropna().unique())
    _missing_for_us = _non_us_mc - _us_mc

    # [FIX-8] _us_table_cols skip 제거: _missing_for_us = _non_us_mc - _us_mc 만으로 판단
    # US 테이블이 매핑에 있어도 실제 데이터가 없으면 dummy가 삽입되어야 함
    # (US 전용 column은 _non_us_mc에 없으므로 _missing_for_us에 포함되지 않아 문제없음)

    if _missing_for_us:
        _us_rows = union_df[_us_mask]
        if _us_rows.empty:
            print("[WARN] No US data → skip US dummy insert")
        else:
            _tmpl = _us_rows.iloc[0].to_dict()
            _dummy_list = []
            for _mc in sorted(_missing_for_us, key=str):

                _ref = union_df.loc[_non_us_mask & (union_df["metric_col"] == _mc)]
                _row = {col: None for col in union_df.columns}
                _row.update(_tmpl)
                _row["metric_col"] = _mc
                _row["metric_value_origin"] = 0.0
                _row["metric_value_adj"]    = 0.0
                if not _ref.empty:
                    for _dc in ["Start_Date", "End_Date"]:
                        if _dc in _ref.columns:
                            _row[_dc] = _ref.iloc[0][_dc]
                _j = split_metric_col(_mc)
                _row.update(_j)
                _row["value"] = _j.get("metric_name", "")
                if "PAID_NONPAID" in union_df.columns:
                    _paid_vals = union_df.loc[
                        _non_us_mask & (union_df["metric_col"] == _mc), "PAID_NONPAID"
                    ]
                    _row["PAID_NONPAID"] = _paid_vals.mode().iloc[0] if not _paid_vals.mode().empty else "-"
                _dummy_list.append(_row)

            if _dummy_list:
                _us_dummy_df = pd.DataFrame(_dummy_list, columns=union_df.columns)
                union_df = pd.concat([union_df, _us_dummy_df], ignore_index=True)
                print(f"US 누락 metric_col dummy {len(_dummy_list)}개 삽입")
            else:
                print("US dummy 삽입 불필요 (모든 metric_col 이미 존재)")

    finalize_df(union_df).to_csv(union_path, index=False, encoding="utf-8-sig", float_format="%.6f")
    print(f"\n▶ union 저장: {union_path} ({len(union_df):,}행)")

# ── 결과 요약 ─────────────────────────────────────────────────────
print(f"\n✅ 완료: {len(ok_list)}개")
for t in ok_list: print(f"  - {t}")
print(f"\n⏭️  리포트 번호 없어서 skip: {len(no_report_list)}개")
for t in no_report_list: print(f"  - {t}")
print(f"\n⚠️  매핑 없어서 skip: {len(skip_list)}개")
for t in skip_list: print(f"  - {t}")
print(f"\n⚠️  매핑컬럼 불일치 skip: {len(no_vars_list)}개")
for t in no_vars_list: print(f"  - {t}")

▶ 환율 컬럼: 2026-01-26(2026) / 2025-01-26(2025)
▶ App 없는 site (53개): ['africa_en', 'africa_fr', 'africa_pt', 'al', 'ar', 'az', 'ba', 'bd', 'be', 'be_fr', 'bg', 'ch', 'ch_fr', 'dk', 'ee', 'eg', 'fi', 'ge', 'gr', 'hk_en', 'hr', 'ie', 'il', 'iq_ar', 'iq_ku', 'iran', 'jp', 'kz_kz', 'kz_ru', 'latin', 'latin_en', 'lb', 'levant', 'levant_ar', 'lt', 'lv', 'mk', 'mm', 'mn', 'n_africa', 'no', 'pk', 'ps', 'pt', 'py', 'rs', 'ru', 'si', 'sk', 'ua', 'uy', 'uz_ru', 'uz_uz']
▶ 처리 대상: 18개 파일 (tb_key별 최신 파일 기준)

▶ Pre-scan 완료
  Global site_codes (27개): ['AE', 'AE_AR', 'AT', 'AU', 'BE', 'CA', 'CA_FR', 'CL', 'CO', 'CZ', 'DE', 'ES', 'FR', 'HU', 'IL', 'IN', 'IT', 'MX', 'MY', 'NL', 'PE', 'PL', 'PT', 'RO', 'SE', 'UK', 'VN']
  US site_codes (0개): []
  [FIX-5 확인] Global/US site 중복 없음 → 더미 삽입 기준 올바름 ✓

  1_1_basic_traffic_cmp → App 없는 site App 데이터 42행 0처리
  1_1_basic_traffic_cmp → year-split: 875행 → 875행 (time계열 평균, 나머지 합산)
  1_1_basic_traffic_cmp → dummy 70행 삽입
  1_1_basic_traffic_cmp_prior → App 없는 site App 데이터 6